# Sprint S19–S25: Búsqueda de Generalización entre Señantes

**Objetivo:** F1-test ≥ 0.35 **Y** HE3 ✅ PASA simultáneamente  
**Dataset:** `dataset_s15.npz` — 12,259 muestras, 101 clases (min_muestras=15), LSP+AEC  
**Arquitectura base:** proj(→128) → LayerNorm → BiLSTM(128,256) → TemporalAttention → head  
**HPs (S13-best):** hidden=256, n_layers=1, dropout=0.20, lr=1.709e-3, wd=1.296e-4, ls=0.15  

## Criterio HE3 (Holdout de señantes no vistos)
```
ok_f1  = |F1_test - F1_holdout| ≤ 0.15
ok_psi =  PSI < 0.20
ok_ks  =  KS p-value > 0.05
```
GroupShuffleSplit(SEED=42, test_size=0.20) → holdout fijo: dgi156(932)+vineta(843)+abecedario(764)+glosa(2) = 2,541 muestras

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Resultados de todos los sprints S19-S25
data = [
    {'Sprint':'S19', 'Features':'Posición 150D (z-score global)',     'F1_test':0.3357, 'F1_holdout':0.0260, 'Delta_F1':0.3097, 'KS_p':0.0000, 'PSI':0.0877, 'HE3':'❌'},
    {'Sprint':'S20', 'Features':'DANN 71 grupos señantes',             'F1_test':0.2236, 'F1_holdout':0.0118, 'Delta_F1':0.2118, 'KS_p':0.0000, 'PSI':0.0442, 'HE3':'❌'},
    {'Sprint':'S21', 'Features':'Body-centered (BUG renorm)',          'F1_test':0.0833, 'F1_holdout':0.0047, 'Delta_F1':0.0787, 'KS_p':0.0000, 'PSI':0.0154, 'HE3':'✅*'},
    {'Sprint':'S22', 'Features':'Body-centered 150D (sin renorm)',     'F1_test':0.3163, 'F1_holdout':0.0246, 'Delta_F1':0.2917, 'KS_p':0.0000, 'PSI':0.1741, 'HE3':'❌'},
    {'Sprint':'S23', 'Features':'Velocidad 150D (frame-a-frame)',      'F1_test':0.1778, 'F1_holdout':0.0119, 'Delta_F1':0.1659, 'KS_p':0.0358, 'PSI':0.1247, 'HE3':'❌'},
    {'Sprint':'S24', 'Features':'Manos-only 84D (relativas a nariz)', 'F1_test':0.2551, 'F1_holdout':0.0197, 'Delta_F1':0.2354, 'KS_p':0.0000, 'PSI':0.1596, 'HE3':'❌'},
    {'Sprint':'S25', 'Features':'Source-DANN 6 dominios',             'F1_test':0.2262, 'F1_holdout':0.0118, 'Delta_F1':0.2144, 'KS_p':0.0000, 'PSI':0.0758, 'HE3':'❌'},
]
df = pd.DataFrame(data)
df['Ratio_Delta_F1'] = (df['Delta_F1'] / df['F1_test']).round(3)
print(df[['Sprint','F1_test','F1_holdout','Delta_F1','Ratio_Delta_F1','HE3']].to_string(index=False))
print('\n* S21 pasa HE3 trivialmente: el modelo es uniformemente débil (F1=0.083)')

## Hallazgo clave: Ratio ΔF1/F1-test ≈ 0.92 (constante)

Sin importar la representación (posición, velocidad, manos-only, DANN), el ratio se mantiene ~0.92.  
Esto es una **restricción estructural del dataset**, no del modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_plot = df[df['Sprint'] != 'S21']  # excluir S21 trivial
colors = ['#2196F3','#9C27B0','#FF5722','#4CAF50','#FF9800','#E91E63']

# Plot 1: F1-test vs Delta_F1
ax = axes[0]
for i, row in df_plot.iterrows():
    ax.scatter(row['F1_test'], row['Delta_F1'], s=120, color=colors[list(df_plot.index).index(i)], zorder=5)
    ax.annotate(row['Sprint'], (row['F1_test'], row['Delta_F1']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)

x_range = np.linspace(0.10, 0.40, 100)
ax.plot(x_range, 0.92 * x_range, 'r--', alpha=0.7, label='ratio=0.92 (observado)')
ax.axhline(0.15, color='green', linestyle=':', linewidth=2, label='umbral HE3 (ΔF1≤0.15)')
ax.axvline(0.35, color='blue', linestyle=':', linewidth=2, label='objetivo F1-test≥0.35')
ax.set_xlabel('F1-test', fontsize=12)
ax.set_ylabel('ΔF1 (Delta)', fontsize=12)
ax.set_title('F1-test vs ΔF1: ratio constante ~0.92', fontsize=13)
ax.legend(fontsize=9)
ax.set_xlim(0.10, 0.42); ax.set_ylim(0.10, 0.35)
ax.fill_between([0.35, 0.42], [0, 0], [0.15, 0.15], alpha=0.15, color='green', label='zona objetivo')
ax.grid(True, alpha=0.3)

# Plot 2: F1-holdout comparison
ax2 = axes[1]
bars = ax2.bar(df['Sprint'], df['F1_holdout'], color=['#2196F3','#9C27B0','gray','#FF5722','#4CAF50','#FF9800','#E91E63'])
ax2.axhline(0.185, color='green', linestyle='--', linewidth=2, label='holdout mínimo para HE3\n(si F1-test=0.35, ΔF1≤0.15)')
ax2.set_xlabel('Sprint', fontsize=12)
ax2.set_ylabel('F1 Holdout', fontsize=12)
ax2.set_title('F1 en Holdout de Señantes No Vistos', fontsize=13)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../data/sprint_s19_s25_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
print('F1 holdout máximo alcanzado: S19=0.0260')
print('F1 holdout necesario para pasar HE3 con F1-test=0.35: ≥ 0.185 (7.1× más alto)')

## Análisis de la Causa Raíz

### Estructura del holdout HE3 (2,541 muestras, 17 clases)

| Tipo | Clases | Muestras holdout | Core training | Causa del fallo |
|------|--------|-----------------|---------------|------------------|
| **Letras ABC** | A, E, L, N, T, X | 764 (30%) | **0** | Sin datos de entrenamiento |
| **HISTORIAS_VINETAS_3** | HV_3 | 923 (36%) | **0** | Sin datos de entrenamiento |
| **HISTORIAS_VINETAS (×9)** | HV_2,4,6,12,15,17,25,37,42 | 710 (28%) | 32-187 | **Cross-source**: train=dgi156, holdout=vineta (o viceversa) |
| **BIEN** | BIEN | 2 (0.1%) | 15 | Solo 2 muestras holdout |

**El 66% del holdout son clases con 0 muestras de entrenamiento → F1=0 independiente del modelo.**

### Por qué DANN (S20, S25) empeora el holdout:
Las HISTORIAS_VINETAS son casi exclusivamente de una sola fuente por clase en entrenamiento
(ej: HV_12 training=dgi156, HV_12 holdout=vineta). Al hacer el encoder invariante a fuente,
DANN elimina la información de fuente que es la ÚNICA señal discriminativa para esas clases.

In [ ]:
# Análisis matemático del límite teórico
print('=== ANÁLISIS MATEMÁTICO DEL LÍMITE TEÓRICO ===')
print()

n_holdout_total  = 2541
n_zero_training  = 764 + 923  # letras + HV_3
n_crosssource    = 710         # 9 HISTORIAS_VINETAS con entrenamiento
n_bien           = 2

print(f'Holdout total: {n_holdout_total} muestras, 17 clases')
print(f'  Clases sin entrenamiento: 7 clases, {n_zero_training} muestras ({100*n_zero_training/n_holdout_total:.0f}%)')
print(f'  F1 máx posible para estas 7 clases: 0.0  (sin entrenamiento)')
print(f'  Clases cross-source con training: 9+1=10 clases, {n_crosssource+n_bien} muestras')
print()
print('Con macro F1 sobre 17 clases:')
for f1_cross in [0.10, 0.20, 0.315, 0.50, 1.00]:
    macro_holdout = 10 * f1_cross / 17
    delta_if_test_035 = abs(0.35 - macro_holdout)
    print(f'  Si HISTORIAS_VINETAS F1={f1_cross:.2f}: '
          f'holdout_macro={macro_holdout:.3f}, ΔF1={delta_if_test_035:.3f} '
          f"{'✅ PASA' if delta_if_test_035 <= 0.15 else '❌ FALLA'}")

print()
print(f'HISTORIAS_VINETAS F1 actual (promedio observado): ~0.044')
print(f'HISTORIAS_VINETAS F1 necesario: ≥ 0.315  (7.2× mejora requerida)')
print()
print('Ningún enfoque de feature engineering rompe este límite estructural.')

## Mejor Modelo: S19

Sin HE3, el mejor modelo es **S19** (F1-test=0.3357, el más cercano al objetivo 0.35).

- Checkpoint: `checkpoints/bilstm_s19.pt`  
- Features: posición global z-score, [30, 150]  
- Augmentación: noise σ=0.020, scale ±15%, flip 70%, time-warp 60%, drop 50%  

## Recomendaciones para Superar HE3

1. **Añadir datos de letras A,E,L,N,T,X** de otras fuentes (no abecedario) al dataset de entrenamiento  
2. **Rebalancear la estructura de grupos** en GroupShuffleSplit para que las clases importantes tengan muestras en training  
3. **Aumentar diversidad de señantes** por clase: conseguir múltiples firmantes por señal de video  
4. **Evaluar con GroupShuffleSplit diferente seed** para validar si el problema es específico de este split  

Sin estos cambios, el ratio ΔF1/F1-test ≈ 0.92 impide cumplir el doble objetivo.

In [ ]:
# Resumen final
print('=== RESUMEN SPRINT S19-S25 ===')
print()
print(f'{"Sprint":<8} {"F1-test":<10} {"F1-hold":<10} {"ΔF1":<8} {"KS-p":<8} {"HE3"}')
print('-' * 55)
for row in data:
    print(f"{row['Sprint']:<8} {row['F1_test']:.4f}     {row['F1_holdout']:.4f}     "
          f"{row['Delta_F1']:.4f}   {row['KS_p']:.4f}   {row['HE3']}")
print()
print('Mejor F1-test   : S19 = 0.3357  (vs objetivo 0.35)  [-0.55%]')
print('Menor ΔF1       : S23 = 0.1659  (vs umbral  0.15)  [+10.6%]')
print('Objetivo F1≥0.35 + HE3✅: NO ALCANZADO (restricción estructural del dataset)')